# NLP with Machine Learning Assignments

## 1. Sentiment Analysis

1. Create a new _nlp_machine_learning_ environment
2. Launch Jupyter Notebook
3. Read in the _movie_reviews.csv_ file
4. Apply sentiment analysis to the _movie_info_ column
5. Sort the sentiment scores to return the top 10 and bottom 10 sentiment scores and their corresponding movie titles

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('../Data/movie_reviews.csv')

In [4]:
df_sentiment_analysis = df[['movie_title', 'movie_info']].copy()

In [5]:
df_sentiment_analysis

,movie_title,movie_info
0,A Dog's Journey,Bailey (voiced again by Josh Gad) is living th...
1,A Dog's Way Home,"Separated from her owner, a dog sets off on an..."
2,A Tuba to Cuba,The leader of New Orleans' famed Preservation ...
3,A Vigilante,"A once abused woman, Sadie (Olivia Wilde), dev..."
4,After,Based on Anna Todd's best-selling novel which ...
...,...,...
161,Velvet Buzzsaw,Velvet Buzzsaw is a satirical thriller set in ...
162,What Men Want,Inspired by the Nancy Meyers hit romantic come...
163,Wild Rose,WILD ROSE tells the complicated story of Rose-...
164,Wine Country,In honor of Rebecca (Rachel Dratch)'s 50th bir...


In [6]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyser = SentimentIntensityAnalyzer()

In [7]:
def get_sentiment(text: str):
    analyser = SentimentIntensityAnalyzer()
    output = analyser.polarity_scores(text)['compound']
    return output

In [8]:
df_sentiment_analysis['sentiment'] = df_sentiment_analysis['movie_info'].apply(get_sentiment)

In [9]:
df_sentiment_analysis.head()

,movie_title,movie_info,sentiment
0,A Dog's Journey,Bailey (voiced again by Josh Gad) is living th...,0.9837
1,A Dog's Way Home,"Separated from her owner, a dog sets off on an...",0.9237
2,A Tuba to Cuba,The leader of New Orleans' famed Preservation ...,0.9360
3,A Vigilante,"A once abused woman, Sadie (Olivia Wilde), dev...",-0.0334
4,After,Based on Anna Todd's best-selling novel which ...,0.9349


In [10]:
df_sentiment_analysis.sort_values(by='sentiment', ascending=False).head(10)

,movie_title,movie_info,sentiment
23,Breakthrough,BREAKTHROUGH is based on the inspirational tru...,0.9915
81,Missing Link,"This April, meet Mr. Link (Galifianakis): 8 fe...",0.9909
130,The Laundromat,When her idyllic vacation takes an unthinkable...,0.9908
48,Five Feet Apart,Stella Grant (Haley Lu Richardson) is every bi...,0.9889
156,UglyDolls,"In the adorably different town of Uglyville, w...",0.9862
93,Red Joan,"In a picturesque village in England, Joan Stan...",0.9848
49,Giant Little Ones,Franky Winter (Josh Wiggins) and Ballas Kohl (...,0.9839
0,A Dog's Journey,Bailey (voiced again by Josh Gad) is living th...,0.9837
36,Dumbo,"From Disney and visionary director Tim Burton,...",0.9801
71,Long Shot,Fred Flarsky (Seth Rogen) is a gifted and free...,0.9778


In [11]:
df_sentiment_analysis.sort_values(by='sentiment', ascending=True).head(10)

,movie_title,movie_info,sentiment
7,All Is True,"The year is 1613, Shakespeare is acknowledged ...",-0.9955
148,The Wind,An unseen evil haunts the homestead in this ch...,-0.9838
83,Nightmare Cinema,"In this twisted horror anthology, five strange...",-0.9756
154,Triple Threat,"TRIPLE THREAT, the newest feature from Johnson...",-0.9696
11,Angel of Mine,Noomi Rapace (The Girl with the Dragon Tattoo)...,-0.9687
27,Charlie Says,Three young women were sentenced to death for ...,-0.9643
113,The Curse of La Llorona,"In 1970s Los Angeles, La Llorona is stalking t...",-0.9628
87,Pet Sematary,Based on the seminal horror novel by Stephen K...,-0.9590
142,The Standoff at Sparrow Creek,"After a mass shooting at a police funeral, rec...",-0.9590
40,El Chicano,When L.A.P.D. Detective Diego Hernandez is ass...,-0.9578


## 2. Text Classification

1. Clean and normalize the _movie_info_ column using the _maven_text_preprocessing.py_ module
2. Create a Count Vectorizer
* Remove stop words
* Set the minimum document frequency to 10%
3. Create a Naïve Bayes model and a Logistic Regression model to predict which movies are directed by women vs men using the CV
4. Compare their accuracy scores and classification reports
5. Using the better performing model, return the top 5 movies that the model predicts are most likely directed by a women

In [12]:
# decrease the column width
pd.set_option('display.max_colwidth', 100)

# view movie reviews
df.head(2)

,movie_title,rating,genre,in_theaters_date,movie_info,directors,director_gender,tomatometer_rating,audience_rating,critics_consensus
0,A Dog's Journey,PG,"Drama, Kids & Family",5/17/19,"Bailey (voiced again by Josh Gad) is living the good life on the Michigan farm of his ""boy,"" Eth...",Gail Mancuso,female,50,92,"A Dog's Journey is as sentimental as one might expect, but even cynical viewers may find their a..."
1,A Dog's Way Home,PG,Drama,1/11/19,"Separated from her owner, a dog sets off on an 400-mile journey to get back to the safety and se...",Charles Martin Smith,male,60,71,"A Dog's Way Home may not quite be a family-friendly animal drama fan's best friend, but this can..."


In [14]:
# import the text preprocessing steps we created in the last section
import maven_preprocessing_text

# apply them to the reviews
df['text_clean'] = maven_preprocessing_text.clean_and_normalize(df.movie_info)
df.head(2)

,movie_title,rating,genre,in_theaters_date,movie_info,directors,director_gender,tomatometer_rating,audience_rating,critics_consensus,text_clean
0,A Dog's Journey,PG,"Drama, Kids & Family",5/17/19,"Bailey (voiced again by Josh Gad) is living the good life on the Michigan farm of his ""boy,"" Eth...",Gail Mancuso,female,50,92,"A Dog's Journey is as sentimental as one might expect, but even cynical viewers may find their a...",bailey voice josh gad live good life michigan farm boy ethan dennis quaid ethans wife hannah mar...
1,A Dog's Way Home,PG,Drama,1/11/19,"Separated from her owner, a dog sets off on an 400-mile journey to get back to the safety and se...",Charles Martin Smith,male,60,71,"A Dog's Way Home may not quite be a family-friendly animal drama fan's best friend, but this can...",separate owner dog set 400mile journey safety security place call home way meet series new frien...


In [15]:
# import libraries
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

In [16]:
# create a count vectorizer matrix
cv = CountVectorizer(stop_words='english', min_df=.1)
X = cv.fit_transform(df.text_clean)

In [17]:
# view the features / inputs X
X_df = pd.DataFrame(X.toarray(), columns=cv.get_feature_names_out())
X_df.head()

,begin,discover,family,film,follow,force,friend,home,leave,life,...,man,new,set,star,story,turn,woman,world,year,young
0,1,0,0,0,0,0,1,0,1,3,...,0,2,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,1,1,0,1,...,0,1,1,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,2,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
4,0,0,0,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0


In [18]:
# view the target / output y
y = df.director_gender
y.head()

0    female
1      male
2      male
3    female
4    female
Name: director_gender, dtype: str

In [19]:
# view the number of directors of each gender
y.value_counts()

director_gender
male      134
female     32
Name: count, dtype: int64

In [20]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.2, random_state=42)

# Naive Bayes model
nb = MultinomialNB()
nb.fit(X_train, y_train)

# Predict
y_pred_nb = nb.predict(X_test)

# Evaluate
print(classification_report(y_test, y_pred_nb))
print("Accuracy:", accuracy_score(y_test, y_pred_nb))

              precision    recall  f1-score   support

      female       0.25      0.20      0.22         5
        male       0.87      0.90      0.88        29

    accuracy                           0.79        34
   macro avg       0.56      0.55      0.55        34
weighted avg       0.78      0.79      0.78        34

Accuracy: 0.7941176470588235


In [21]:
# Logistic Regression model
lr = LogisticRegression()
lr.fit(X_train, y_train)

# Predict
y_pred_lr = lr.predict(X_test)

# Evaluate
print(classification_report(y_test, y_pred_lr))
print("Accuracy:", accuracy_score(y_test, y_pred_lr))

              precision    recall  f1-score   support

      female       0.29      0.40      0.33         5
        male       0.89      0.83      0.86        29

    accuracy                           0.76        34
   macro avg       0.59      0.61      0.60        34
weighted avg       0.80      0.76      0.78        34

Accuracy: 0.7647058823529411


## 3. Topic Modeling

1. Using the same preprocessed data as the last assignment, create a Tfidf Vectorizer
* Remove stop words
* Start with min_df = 0.05 and max_df=0.2
2. Create an NMF model to find the main topics in the movie descriptions
* Start with n_components=2
3. Tweak the model by updating the Tfidf Vectorizer parameters and number of topics
4. Interpret and name the topics
5. For two of the topics, return the top movies that contain the topic